In [2]:
import re
from collections import defaultdict

import cv2
import numpy as np

import onnxruntime as ort
from ultralytics import YOLO
from fast_alpr import ALPR

In [3]:
VIDEO_PATH  = "video.mp4"
OUTPUT_PATH = "output_plaka.mp4"
DEVICE      = "cpu"

VEHICLE_CLASSES = [2, 5, 7]
VEHICLE_CONF    = 0.40

ZONE_TOP_RATIO    = 0.70
ZONE_BOTTOM_RATIO = 0.90
N_VALID_READS     = 5
PLATE_MIN_CONF    = 0.50

SHOW_WINDOW = False

BOX_THICK   = 5
ZONE_THICK  = 3
FONT_SCALE  = 2.2
FONT_THICK  = 5

COLOR_PLATE = (211, 0, 148)
COLOR_TEXT  = (255, 255, 255)
COLOR_ZONE  = (255, 0, 0)

In [4]:
vehicle_model = YOLO("yolo11m.pt")

try:
    ort.preload_dlls()
except Exception as e:
    print("preload_dlls atlandi:", e)

alpr = ALPR(
    detector_model="yolo-v9-t-384-license-plate-end2end",
    ocr_model="cct-s-v2-global-model",
)

INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['AzureExecutionProvider', 'CPUExecutionProvider'] provider(s)


In [5]:
def normalize_plate(text):
    return re.sub(r"[^A-Z0-9]", "", (text or "").upper())

def is_valid_eu_plate(text):
    t = normalize_plate(text)
    if not (5 <= len(t) <= 9):
        return False
    if not any(c.isdigit() for c in t):
        return False
    if not any(c.isalpha() for c in t):
        return False
    return True

In [6]:
def yeni_durum():
    return {
        "candidates": [],
        "locked_text": None,
        "locked_conf": None,
        "last_plate_box": None,
        "running_text": None,
    }

track_state = defaultdict(yeni_durum)

In [7]:
def en_iyi_plaka(car_crop):
    """car_crop uzerinde fast-alpr calistir, en yuksek OCR confidence'li
    GECERLI plakayi dondur.

    Donen: (text, conf, (px1, py1, px2, py2)) | None
    Koordinatlar crop'a goredir; cizim sirasinda araç offset'i (x1, y1)
    eklenerek full-frame'e tasiniyor."""
    if car_crop is None or car_crop.size == 0:
        return None

    results = alpr.predict(car_crop)
    best = None
    for r in results:
        if r.ocr is None:
            continue
        if not is_valid_eu_plate(r.ocr.text):
            continue

        c = r.ocr.confidence
        conf = (float(np.mean(c))
                if isinstance(c, (list, tuple, np.ndarray)) and len(c)
                else float(c))

        bb = r.detection.bounding_box
        box = (int(bb.x1), int(bb.y1), int(bb.x2), int(bb.y2))

        if best is None or conf > best[1]:
            best = (normalize_plate(r.ocr.text), conf, box)
    return best

In [8]:
def kutu_ciz(frame, box, color, thickness=None):
    if thickness is None:
        thickness = BOX_THICK
    x1, y1, x2, y2 = box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

def etiket_ciz(frame, text, org, bg_color, text_color=(0, 0, 0)):
    x, y = org
    y = max(y, int(36 * FONT_SCALE))
    (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX,
                                   FONT_SCALE, FONT_THICK)
    pad = int(6 * FONT_SCALE)
    cv2.rectangle(frame, (x, y - th - bl - pad), (x + tw + pad, y), bg_color, -1)
    cv2.putText(frame, text, (x + pad // 2, y - bl - pad // 3),
                cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, text_color,
                FONT_THICK, cv2.LINE_AA)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

zone_y_top    = int(height * ZONE_TOP_RATIO)
zone_y_bottom = int(height * ZONE_BOTTOM_RATIO)

writer = cv2.VideoWriter(
    OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
)

frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1

    clean = frame.copy()

    results = vehicle_model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",
        classes=VEHICLE_CLASSES,
        conf=VEHICLE_CONF,
        device=DEVICE,
        verbose=False,
    )[0]

    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids   = results.boxes.id.cpu().numpy().astype(int)

        for (bx1, by1, bx2, by2), tid in zip(boxes, ids):
            x1, y1, x2, y2 = int(bx1), int(by1), int(bx2), int(by2)
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            st = track_state[tid]
            in_zone = zone_y_top <= cy <= zone_y_bottom

            if in_zone:
                cx1, cy1 = max(0, x1), max(0, y1)
                cx2, cy2 = max(0, x2), max(0, y2)
                crop = clean[cy1:cy2, cx1:cx2]
                best = en_iyi_plaka(crop)
                if best is not None:
                    ptext, pconf, (px1, py1, px2, py2) = best
                    st["last_plate_box"] = (cx1 + px1, cy1 + py1,
                                            cx1 + px2, cy1 + py2)
                    if st["locked_text"] is None:
                        st["running_text"] = ptext
                        if pconf >= PLATE_MIN_CONF:
                            st["candidates"].append((ptext, pconf))
                            if len(st["candidates"]) >= N_VALID_READS:
                                best_c = max(st["candidates"],
                                             key=lambda c: c[1])
                                st["locked_text"], st["locked_conf"] = best_c

            if in_zone and st["last_plate_box"] is not None:
                kutu_ciz(frame, st["last_plate_box"], COLOR_PLATE)

            if st["locked_text"] is not None:
                etiket_ciz(frame, st["locked_text"], (x1, y1),
                           COLOR_PLATE, COLOR_TEXT)

    cv2.line(frame, (0, zone_y_top), (width, zone_y_top), COLOR_ZONE, ZONE_THICK)
    cv2.line(frame, (0, zone_y_bottom), (width, zone_y_bottom), COLOR_ZONE, ZONE_THICK)

    writer.write(frame)
    if SHOW_WINDOW:
        cv2.imshow("ALPR", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
writer.release()
if SHOW_WINDOW:
    cv2.destroyAllWindows()
print(f"Bitti -> {OUTPUT_PATH}")